# Notebook 02 (v2): Feature Extraction with stronger features
**AID 427 Project Phase 1: MultiCaRe-Fusion**






In [1]:
import os, sys, json, time, zipfile, warnings, subprocess
import numpy as np, pandas as pd
from PIL import Image
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
SEED, BATCH = 42, 128
USE_RESNET, USE_DINOV2, USE_BIOMEDCLIP, USE_TEXT_EMB, TTA = True, True, True, True, True
TFIDF_CAPTION_FEATURES, TFIDF_CASE_FEATURES = 3000, 3000

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "transformers"], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/MultiCaRe"
    IMG_ROOT = "/content/images"
    if not os.path.isdir(IMG_ROOT):
        os.makedirs(IMG_ROOT, exist_ok=True)
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, "subset_images.zip")) as z: z.extractall(IMG_ROOT)
    SUBSET, FEAT_DIR = os.path.join(DRIVE_DIR, "subset.parquet"), os.path.join(DRIVE_DIR, "features")
else:
    DATA_DIR = next(p for p in ["../DataSet", "DataSet", "."] if os.path.exists(os.path.join(p, "outputs", "subset.parquet")))
    IMG_ROOT, SUBSET, FEAT_DIR = DATA_DIR, os.path.join(DATA_DIR, "outputs", "subset.parquet"), os.path.join(DATA_DIR, "outputs", "features")
os.makedirs(FEAT_DIR, exist_ok=True)
df = pd.read_parquet(SUBSET).reset_index(drop=True)
df["img_path"] = [os.path.join(IMG_ROOT, f[:4], f[:5], f) for f in df.file]
print("Colab:", IN_COLAB, "| rows:", len(df), "| images found:", np.mean([os.path.exists(p) for p in df.img_path[:500]]))

Mounted at /content/drive
Colab: True | rows: 21988 | images found: 1.0


## 1. Labels, row index, tabular features

In [2]:
from sklearn.preprocessing import LabelEncoder
le_type, le_sub = LabelEncoder().fit(df.image_type), LabelEncoder().fit(df.image_subtype)
y_type, y_sub = le_type.transform(df.image_type), le_sub.transform(df.image_subtype)
np.save(os.path.join(FEAT_DIR, "y_type.npy"), y_type); np.save(os.path.join(FEAT_DIR, "y_subtype.npy"), y_sub)
json.dump({"image_type": list(le_type.classes_), "image_subtype": list(le_sub.classes_)}, open(os.path.join(FEAT_DIR, "classes.json"), "w"), indent=1)
df[["file_id", "file", "article_id", "patient_id", "image_type", "image_subtype", "fold", "split"]].to_csv(os.path.join(FEAT_DIR, "index.csv"), index=False)
tab = pd.concat([df[["age_imputed", "age_missing"]].reset_index(drop=True), pd.get_dummies(df.gender_clean, prefix="gender").astype(int), pd.get_dummies(df.age_group.astype(str), prefix="agegrp").astype(int)], axis=1)
np.save(os.path.join(FEAT_DIR, "tab_features.npy"), tab.values.astype(np.float32)); json.dump(list(tab.columns), open(os.path.join(FEAT_DIR, "tab_columns.json"), "w"))
print("classes:", len(le_type.classes_), len(le_sub.classes_), "| tabular:", tab.shape)

classes: 7 27 | tabular: (21988, 11)


## 2. TF-IDF text features (fitted on training rows only)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
train_mask = (df.split == "train").values
def tfidf(col, nfeat, name):
    vec = TfidfVectorizer(max_features=nfeat, ngram_range=(1, 2), min_df=3, sublinear_tf=True, stop_words="english").fit(df.loc[train_mask, col].fillna(""))
    X = vec.transform(df[col].fillna("")).astype(np.float32); sp.save_npz(os.path.join(FEAT_DIR, f"{name}.npz"), X)
    json.dump(list(vec.get_feature_names_out()), open(os.path.join(FEAT_DIR, f"{name}_vocab.json"), "w")); print(name, X.shape); return X
X_cap, X_case = tfidf("caption_clean", TFIDF_CAPTION_FEATURES, "text_caption_tfidf"), tfidf("case_text_short", TFIDF_CASE_FEATURES, "text_case_tfidf")

text_caption_tfidf (21988, 3000)
text_case_tfidf (21988, 3000)


## 3. Handcrafted image features (colour/intensity statistics + HOG)

In [4]:
from skimage.feature import hog
def handcrafted(path):
    im = Image.open(path).convert("RGB").resize((64, 64)); a = np.asarray(im, dtype=np.float32) / 255.0; g = a.mean(axis=2)
    stats = np.r_[a.mean((0, 1)), a.std((0, 1))]; hist = np.histogram(g, bins=16, range=(0, 1))[0] / g.size
    colourful = np.abs(a[..., 0] - a[..., 1]).mean() + np.abs(a[..., 1] - a[..., 2]).mean()
    return np.r_[stats, hist, colourful, hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)].astype(np.float32)
t0 = time.time(); H = []
for i, p in enumerate(df.img_path):
    try: H.append(handcrafted(p))
    except Exception: H.append(np.zeros(6 + 16 + 1 + 324, dtype=np.float32))
    if i % 4000 == 0: print(i, "/", len(df), f"{time.time()-t0:.0f}s")
H = np.vstack(H); np.save(os.path.join(FEAT_DIR, "img_handcrafted.npy"), H); print("handcrafted:", H.shape)

0 / 21988 0s
4000 / 21988 30s
8000 / 21988 59s
12000 / 21988 87s
16000 / 21988 115s
20000 / 21988 145s
handcrafted: (21988, 347)


## 4. Deep image embeddings: ResNet50, DINOv2, BiomedCLIP (one pass over the images, with padding and mirror TTA)

In [5]:
HAS_TORCH = True
try:
    import torch, torchvision
    from torchvision import transforms as T
    from torch.utils.data import Dataset, DataLoader
except Exception as e:
    HAS_TORCH = False; print("PyTorch not available -> deep embeddings skipped:", e)
saved_blocks = []
if HAS_TORCH:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device, "| torch", torch.__version__)
    if device == "cpu": print("WARNING: no GPU. On Colab choose Runtime > Change runtime type > GPU, otherwise this is very slow.")
    class PadResize:
        def __init__(self, size=224): self.size = size
        def __call__(self, im):
            w, h = im.size; s = self.size / max(w, h); im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
            canvas = Image.new("RGB", (self.size, self.size), (0, 0, 0)); canvas.paste(im, ((self.size - im.width) // 2, (self.size - im.height) // 2)); return canvas
    tf = T.Compose([PadResize(224), T.ToTensor()])                      # values in [0,1]; each backbone normalises on the GPU
    class DS(Dataset):
        def __init__(self, paths): self.paths = list(paths)
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            try: return tf(Image.open(self.paths[i]).convert("RGB"))
            except Exception: return torch.zeros(3, 224, 224)
    dl = DataLoader(DS(df.img_path), batch_size=BATCH, shuffle=False, num_workers=(2 if IN_COLAB else 0))
    def tens(v): return torch.tensor(v, device=device).view(1, 3, 1, 1)
    IMN_M, IMN_S = tens([0.485, 0.456, 0.406]), tens([0.229, 0.224, 0.225])
    extractors = {}; HUB = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"; bm = None
    if USE_RESNET:
        try:
            net = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2); net.fc = torch.nn.Identity(); net = net.eval().to(device)
            extractors["resnet50"] = lambda x: net((x - IMN_M) / IMN_S)
        except Exception as e: print("ResNet50 failed:", repr(e)[:200])
    if USE_DINOV2:
        try:
            from transformers import AutoModel
            dino = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(device)
            def dino_f(x):
                h = dino(pixel_values=(x - IMN_M) / IMN_S).last_hidden_state
                return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)          # CLS token + mean of patch tokens (1536)
            extractors["dinov2"] = dino_f
        except Exception as e: print("DINOv2 failed:", repr(e)[:200])
    if USE_BIOMEDCLIP:
        try:
            import open_clip
            bm, _, bpre = open_clip.create_model_and_transforms(HUB); bm = bm.eval().to(device)
            nm = [t for t in bpre.transforms if isinstance(t, T.Normalize)][0]; BM_M, BM_S = tens(list(nm.mean)), tens(list(nm.std))
            extractors["biomedclip"] = lambda x: bm.encode_image((x - BM_M) / BM_S)
        except Exception as e: print("BiomedCLIP failed:", repr(e)[:200]); bm = None
    print("backbones loaded:", list(extractors))
    feats = {k: [] for k in extractors}; t0 = time.time()
    with torch.no_grad():
        for b, x in enumerate(dl):
            x = x.to(device)
            for k, f in extractors.items():
                with torch.autocast(device_type=device, enabled=(device == "cuda")):
                    z = f(x)
                    if TTA: z = (z + f(torch.flip(x, dims=[3]))) / 2
                feats[k].append(z.float().cpu().numpy())
            if b % 20 == 0: print(f"batch {b}/{len(dl)}  {time.time()-t0:.0f}s")
    for k, v in feats.items():
        arr = np.vstack(v).astype(np.float32); np.save(os.path.join(FEAT_DIR, f"img_{k}.npy"), arr); saved_blocks.append(f"img_{k}"); print("saved", k, arr.shape)

device: cuda | torch 2.14.1+cu130
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

backbones loaded: ['resnet50', 'dinov2', 'biomedclip']
batch 0/172  5s
batch 20/172  41s
batch 40/172  78s
batch 60/172  118s
batch 80/172  162s
batch 100/172  204s
batch 120/172  246s
batch 140/172  289s
batch 160/172  331s
saved resnet50 (21988, 2048)
saved dinov2 (21988, 1536)
saved biomedclip (21988, 512)


## 5. BiomedCLIP text embeddings (caption and case text)

In [6]:
if HAS_TORCH and USE_TEXT_EMB and bm is not None:
    tok = open_clip.get_tokenizer(HUB)
    def embed_text(texts, bs=256):
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), bs):
                t = tok(list(texts[i:i + bs]), context_length=256).to(device)
                with torch.autocast(device_type=device, enabled=(device == "cuda")): z = bm.encode_text(t)
                out.append(z.float().cpu().numpy())
        return np.vstack(out).astype(np.float32)
    t0 = time.time()
    for name, col in [("txt_biomedclip_caption", "caption"), ("txt_biomedclip_case", "case_text_short")]:
        arr = embed_text(df[col].fillna("").tolist()); np.save(os.path.join(FEAT_DIR, f"{name}.npy"), arr); saved_blocks.append(name); print("saved", name, arr.shape, f"{time.time()-t0:.0f}s")
else: print("text embeddings skipped (needs PyTorch and a loaded BiomedCLIP)")
json.dump(saved_blocks, open(os.path.join(FEAT_DIR, "saved_blocks.json"), "w"))

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

saved txt_biomedclip_caption (21988, 512) 95s
saved txt_biomedclip_case (21988, 512) 212s


## 6. Sanity check: signal per feature block
Logistic Regression per block, trained on train rows and tested on test rows (target = `image_subtype`). Not the final model: only a check that every block carries signal and which ones are strongest.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
tr, te = train_mask, ~train_mask
blocks = {"handcrafted": H, "tabular": tab.values.astype(np.float32)}
for f in sorted(os.listdir(FEAT_DIR)):
    if f.endswith(".npy") and (f.startswith("img_") or f.startswith("txt_")) and f != "img_handcrafted.npy": blocks[f[:-4]] = np.load(os.path.join(FEAT_DIR, f))
for nm, X in [("caption TF-IDF (SVD 200)", X_cap), ("case TF-IDF (SVD 200)", X_case)]: blocks[nm] = TruncatedSVD(min(200, X.shape[1] - 1), random_state=SEED).fit(X[tr]).transform(X)
res = {}
for name, X in blocks.items():
    sc = StandardScaler().fit(X[tr]); clf = LogisticRegression(max_iter=300, C=0.05).fit(sc.transform(X[tr]), y_sub[tr]); res[name] = round(clf.score(sc.transform(X[te]), y_sub[te]), 3)
print("test accuracy for image_subtype (chance ~ %.3f), by feature block:" % (np.bincount(y_sub[te]).max() / te.sum()))
pd.Series(res).sort_values(ascending=False)

test accuracy for image_subtype (chance ~ 0.050), by feature block:


,0
img_biomedclip,0.776
img_dinov2,0.754
img_resnet50,0.730
txt_biomedclip_caption,0.689
caption TF-IDF (SVD 200),0.618
handcrafted,0.486
txt_biomedclip_case,0.328
case TF-IDF (SVD 200),0.318
tabular,0.075


In [8]:
rows = [(f, round(os.path.getsize(os.path.join(FEAT_DIR, f)) / 1e6, 1)) for f in sorted(os.listdir(FEAT_DIR))]
pd.DataFrame(rows, columns=["file", "MB"])

,file,MB
0,classes.json,0.0
1,img_biomedclip.npy,45.0
2,img_dinov2.npy,135.1
3,img_handcrafted.npy,30.5
4,img_resnet50.npy,180.1
5,index.csv,2.5
6,saved_blocks.json,0.0
7,tab_columns.json,0.0
8,tab_features.npy,1.0
9,text_caption_tfidf.npz,1.2
